# Fine-tuning: Classificador de Mensagens de Clientes

Fine-tuning do **BERTimbau** (BERT em português 🇧🇷) para classificar mensagens em:
- **`venda`** → intenção de compra
- **`suporte`** → dúvida ou problema com produto

In [ ]:
# 1. Instalação de dependências
!pip install transformers datasets accelerate scikit-learn --quiet

In [ ]:
# 2. Imports
import json
import urllib.request
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)
print("✅ Imports OK")

In [ ]:
# 3. Download dos datasets do Notion
TREINO_URL = (
    "https://file.notion.so/f/f/08f749ff-d06d-49a8-a488-9846e081b224/"
    "184b9df6-b0f5-4946-b5b1-6093f96d65e8/treino.jsonl"
    "?table=block&id=195395da-5770-8115-ac85-f7eba4d5433f"
    "&spaceId=08f749ff-d06d-49a8-a488-9846e081b224"
    "&expirationTimestamp=1779955200000"
    "&signature=4PnqK8ibazX0SPVlqFxQWyI4uKURBen_VUcd228BYj8"
    "&downloadName=treino.jsonl"
)
TESTE_URL = (
    "https://file.notion.so/f/f/08f749ff-d06d-49a8-a488-9846e081b224/"
    "2a70b07d-6587-4274-9f31-af5775845f55/teste.jsonl"
    "?table=block&id=195395da-5770-81c5-a72c-fd32bbef40b2"
    "&spaceId=08f749ff-d06d-49a8-a488-9846e081b224"
    "&expirationTimestamp=1779955200000"
    "&signature=z3S2QCNSqPf_fUOtWpSHHa7ipjpkZhtkQyiDGYO6RkA"
    "&downloadName=teste.jsonl"
)
urllib.request.urlretrieve(TREINO_URL, "treino.jsonl")
urllib.request.urlretrieve(TESTE_URL, "teste.jsonl")
print("✅ Arquivos baixados!")

In [ ]:
# 4. Carregamento e exploração dos dados
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)

df_treino = load_jsonl("treino.jsonl")
df_teste  = load_jsonl("teste.jsonl")

print(f"Treino: {len(df_treino)} exemplos")
print(df_treino["completion"].value_counts())
print(f"\nTeste: {len(df_teste)} exemplos")
print(df_teste["completion"].value_counts())
print("\nExemplos:")
df_treino.head()

In [ ]:
# 5. Codificar labels
label2id = {"suporte": 0, "venda": 1}
id2label  = {v: k for k, v in label2id.items()}

df_treino["label"] = df_treino["completion"].map(label2id)
df_teste["label"]  = df_teste["completion"].map(label2id)
print("label2id:", label2id)

In [ ]:
# 6. Tokenização com BERTimbau
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["prompt"], padding="max_length", truncation=True, max_length=128)

ds_treino = Dataset.from_pandas(df_treino[["prompt", "label"]])
ds_teste  = Dataset.from_pandas(df_teste[["prompt", "label"]])

ds_treino = ds_treino.map(tokenize, batched=True)
ds_teste  = ds_teste.map(tokenize, batched=True)

ds_treino = ds_treino.rename_column("label", "labels")
ds_teste  = ds_teste.rename_column("label", "labels")

ds_treino.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
ds_teste.set_format("torch",  columns=["input_ids", "attention_mask", "labels"])

print(f"✅ Tokenização concluída — Treino: {len(ds_treino)} | Teste: {len(ds_teste)}")

In [ ]:
# 7. Carregar modelo para classificação
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)
total = sum(p.numel() for p in model.parameters())
print(f"✅ Modelo carregado | Parâmetros: {total:,}")

In [ ]:
# 8. Métricas
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted"),
    }

In [ ]:
# 9. Treinamento
training_args = TrainingArguments(
    output_dir="./resultados",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=10,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_treino,
    eval_dataset=ds_teste,
    compute_metrics=compute_metrics,
)

print("🚀 Iniciando treinamento...")
trainer.train()

In [ ]:
# 10. Avaliação final
results = trainer.evaluate(ds_teste)
print(f"Accuracy : {results['eval_accuracy']:.4f}")
print(f"F1-Score  : {results['eval_f1']:.4f}")
print(f"Loss      : {results['eval_loss']:.4f}")

In [ ]:
# 11. Relatório de classificação detalhado
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval().to(device)

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in DataLoader(ds_teste, batch_size=32):
        outputs = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        )
        all_preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print(classification_report(all_labels, all_preds, target_names=["suporte", "venda"]))

In [ ]:
# 12. Inferência com frases novas
clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

frases = [
    "Quero comprar uma geladeira duplex",
    "Minha TV não está ligando, o que eu faço?",
    "Vocês têm o micro-ondas 32L em promoção?",
    "O produto chegou com a embalagem danificada",
    "Gostaria de adquirir o aspirador robô modelo X500",
    "Como faço para trocar a peça do ventilador?",
]

print(f"{'Mensagem':<55} {'Classe':<10} {'Confiança'}")
print("-" * 80)
for frase in frases:
    r = clf(frase)[0]
    print(f"{frase:<55} {r['label']:<10} {r['score']:.1%}")

In [ ]:
# 13. Salvar modelo e fazer download
import shutil
model.save_pretrained("./modelo_classificador")
tokenizer.save_pretrained("./modelo_classificador")
print("✅ Modelo salvo em ./modelo_classificador")

shutil.make_archive("modelo_classificador", "zip", "./modelo_classificador")
from google.colab import files
files.download("modelo_classificador.zip")